In [4]:
# 📘 Preprocessing.ipynb
import os
import cv2
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

# === CONFIG ===
VIDEO_DIR = 'Activities'  # 📁 Path to folder: ./Activities/class_name/video.mp4
FRAME_SIZE = (128, 128)
FPS = 10
FRAMES_PER_VIDEO = 10 * FPS  # 10 seconds * 10 FPS = 100 frames
ALLOWED_EXTENSIONS = ('.mp4', '.avi', '.mov', '.mkv')

# === Label Mapping ===
labels = sorted([label for label in os.listdir(VIDEO_DIR) if os.path.isdir(os.path.join(VIDEO_DIR, label))])
label_dict = {label: idx for idx, label in enumerate(labels)}
print("✅ Label Mapping:", label_dict)

# === Video to Frames Preprocessing ===
data = []
targets = []

for label in labels:
    folder_path = os.path.join(VIDEO_DIR, label)
    files = sorted([f for f in os.listdir(folder_path) if f.lower().endswith(ALLOWED_EXTENSIONS)])
    
    for file in tqdm(files, desc=f"📽️ Processing {label}"):
        filepath = os.path.join(folder_path, file)
        cap = cv2.VideoCapture(filepath)
        frames = []
        count = 0

        while cap.isOpened() and count < FRAMES_PER_VIDEO:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.resize(frame, FRAME_SIZE)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
            count += 1
        cap.release()

        # If video shorter than required, pad with last frame
        if len(frames) == 0:
            print(f"⚠️ Skipping empty or unreadable video: {file}")
            continue
        while len(frames) < FRAMES_PER_VIDEO:
            frames.append(frames[-1])
        # If video longer than required, truncate
        frames = frames[:FRAMES_PER_VIDEO]

        data.append(np.array(frames))
        targets.append(label_dict[label])

# === Convert and Save ===
X = np.array(data, dtype=np.uint8)  # shape: (N, 100, 128, 128, 3)
y = np.array(targets, dtype=np.int64)

np.save("X.npy", X)
np.save("y.npy", y)

print("\n✅ Preprocessing Complete")
print("🧠 X shape:", X.shape)   # e.g., (50, 100, 128, 128, 3)
print("🏷️ y shape:", y.shape)   # e.g., (50,)
print("🔖 Classes:", list(label_dict.keys()))


✅ Label Mapping: {'Falling': 0, 'Jumping': 1, 'Lying': 2, 'Running': 3, 'Sitting': 4, 'Standing': 5, 'Walking': 6, 'Walking Downstairs': 7, 'Walking Upstairs': 8}


📽️ Processing Walking Upstairs: 100%|██████████| 6/6 [00:00<00:00,  6.44it/s]



✅ Preprocessing Complete
🧠 X shape: (113, 100, 128, 128, 3)
🏷️ y shape: (113,)
🔖 Classes: ['Falling', 'Jumping', 'Lying', 'Running', 'Sitting', 'Standing', 'Walking', 'Walking Downstairs', 'Walking Upstairs']


In [5]:
import json

# Store label_dict used across models for reproducibility
label_dict = {
    'Falling': 0,
    'Jumping': 1,
    'Lying': 2,
    'Running': 3,
    'Sitting': 4,
    'Standing': 5,
    'Walking': 6,
    'Walking Downstairs': 7,
    'Walking Upstairs': 8
}

# Save label_dict to use across notebooks
with open("/mnt/data/label_dict.json", "w") as f:
    json.dump(label_dict, f)

"/mnt/data/label_dict.json saved ✅"


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/label_dict.json'